# 월마트 고객 데이터 분석 레포트

## Walmart Customer Data Analysis Report

---

## 1. 연구 배경 및 목적

본 연구는 월마트의 약 55만 건의 거래 데이터를 분석하여 **핵심 매출 기여 고객층(Cash Cow)**을 식별하고, 해당 고객층의 **효자 상품(Best Seller)**을 발굴하여 효과적인 마케팅 전략을 도출하는 것을 목적으로 한다.

---

## 2. 연구 가설

### 가설 1

특정 성별과 연령층의 조합이 전체 매출에 가장 큰 기여도를 보일 것이다.

### 가설 2

해당 핵심 세그먼트가 가장 많이 구매하는 특정 제품(효자 상품)이 존재할 것이다.

### 가설 3

핵심 세그먼트 타겟 마케팅은 전체 매출 증대에 가장 효과적일 것이다.

## 3. 데이터 개요

### 3.1 데이터 출처

- Kaggle Walmart Sales Dataset
- 총 550,068건의 거래 데이터
- 5,891명의 고유 고객

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (맥OS)
plt.rcParams['font.family'] = 'AppleGothic'
# 윈도우의 경우: plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드
df = pd.read_csv('/mnt/user-data/uploads/walmart.csv')

In [ ]:
# 데이터 기본 정보 확인
print("=" * 60)
print("데이터 기본 정보")
print("=" * 60)
print(f"총 거래 건수: {len(df):,}건")
print(f"고유 고객 수: {df['User_ID'].nunique():,}명")
print(f"고유 제품 수: {df['Product_ID'].nunique():,}개")
print("=" * 60)

### 3.2 데이터 구조

- 10개 주요 컬럼 (User_ID, Product_ID, Gender, Age, Occupation, City_Category, Stay_In_Current_City_Years, Marital_Status, Product_Category, Purchase)

In [ ]:
# 데이터 구조 확인
print("\n데이터 구조:")
print(df.info())
print("\n데이터 미리보기:")
print(df.head())

### 3.3 데이터 품질 확인

- 결측치 확인
- 이상치 검토
- 데이터 타입 검증

In [ ]:
# 결측치 확인
print("\n결측치 확인:")
print(df.isnull().sum())

# 기술 통계량
print("\n기술 통계량:")
print(df.describe())

---

## 4. 분석 방법론

### 4.1 데이터 전처리

연령대 및 성별 그룹핑, 세그먼트 정의

In [ ]:
# 세그먼트 정의: Gender + Age 조합
df['Segment'] = df['Gender'] + '_' + df['Age']

print("\n생성된 세그먼트:")
print(df['Segment'].unique())
print(f"총 세그먼트 수: {df['Segment'].nunique()}개")

### 4.2 탐색적 데이터 분석 (EDA)

기술 통계량 산출, 분포 확인

### 4.3 세그먼트 분석

성별 × 연령대 교차 분석을 통한 고객 세분화

### 4.4 매출 기여도 분석

세그먼트별 총 매출, 평균 구매액, 구매 건수 비교

### 4.5 제품 분석

핵심 세그먼트의 주요 구매 제품 식별

---

## 5. 분석 결과

### 5.1 전체 고객 인구통계 분포

#### 5.1.1 성별 분포

전체 거래의 성별 비율 분석

In [ ]:
# 성별 분포 분석
gender_dist = df['Gender'].value_counts()
gender_pct = df['Gender'].value_counts(normalize=True) * 100

print("\n성별 분포:")
print(f"남성(M): {gender_dist['M']:,}건 ({gender_pct['M']:.1f}%)")
print(f"여성(F): {gender_dist['F']:,}건 ({gender_pct['F']:.1f}%)")

# 시각화
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
gender_dist.plot(kind='bar', color=['#3498db', '#e74c3c'])
plt.title('성별 거래 건수', fontsize=14, fontweight='bold')
plt.xlabel('성별')
plt.ylabel('거래 건수')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
plt.pie(gender_pct, labels=['남성', '여성'], autopct='%1.1f%%', 
        colors=['#3498db', '#e74c3c'], startangle=90)
plt.title('성별 비율', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#### 5.1.2 연령대 분포

7개 연령층의 거래 건수 분포 분석

In [ ]:
# 연령대 분포 분석
age_order = ['0-17', '18-25', '26-35', '36-45', '46-50', '51-55', '55+']
age_dist = df['Age'].value_counts().reindex(age_order)
age_pct = (df['Age'].value_counts(normalize=True) * 100).reindex(age_order)

print("\n연령대 분포:")
for age in age_order:
    print(f"{age}세: {age_dist[age]:,}건 ({age_pct[age]:.1f}%)")

# 시각화
plt.figure(figsize=(12, 5))
age_dist.plot(kind='bar', color='#2ecc71')
plt.title('연령대별 거래 건수', fontsize=14, fontweight='bold')
plt.xlabel('연령대')
plt.ylabel('거래 건수')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### 5.1.3 성별 × 연령대 조합 현황

총 14개 세그먼트 식별 (7개 연령대 × 2개 성별)

In [ ]:
# 성별 × 연령대 교차표
segment_dist = df.groupby(['Gender', 'Age']).size().unstack(fill_value=0)

print("\n성별 × 연령대 교차표:")
print(segment_dist)

# 히트맵 시각화
plt.figure(figsize=(10, 4))
sns.heatmap(segment_dist[age_order], annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': '거래 건수'})
plt.title('성별 × 연령대 거래 건수 히트맵', fontsize=14, fontweight='bold')
plt.xlabel('연령대')
plt.ylabel('성별')
plt.tight_layout()
plt.show()

### 5.2 매출 기여도 분석

#### 5.2.1 세그먼트별 총 매출액

각 세그먼트의 총 구매금액 합계 및 순위

In [ ]:
# 세그먼트별 총 매출액 계산
segment_sales = df.groupby('Segment')['Purchase'].sum().sort_values(ascending=False)
segment_sales_pct = (segment_sales / segment_sales.sum() * 100)

print("\n세그먼트별 총 매출액 Top 10:")
for i, (seg, sales) in enumerate(segment_sales.head(10).items(), 1):
    print(f"{i}. {seg}: ₹{sales:,.0f} ({segment_sales_pct[seg]:.2f}%)")

# 시각화
plt.figure(figsize=(12, 6))
segment_sales.head(10).plot(kind='barh', color='#9b59b6')
plt.title('세그먼트별 총 매출액 Top 10', fontsize=14, fontweight='bold')
plt.xlabel('총 매출액 (₹)')
plt.ylabel('세그먼트')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

#### 5.2.2 세그먼트별 거래 건수

각 세그먼트의 구매 빈도 분석

In [ ]:
# 세그먼트별 거래 건수
segment_count = df.groupby('Segment').size().sort_values(ascending=False)
segment_count_pct = (segment_count / segment_count.sum() * 100)

print("\n세그먼트별 거래 건수 Top 10:")
for i, (seg, count) in enumerate(segment_count.head(10).items(), 1):
    print(f"{i}. {seg}: {count:,}건 ({segment_count_pct[seg]:.2f}%)")

# 시각화
plt.figure(figsize=(12, 6))
segment_count.head(10).plot(kind='barh', color='#e67e22')
plt.title('세그먼트별 거래 건수 Top 10', fontsize=14, fontweight='bold')
plt.xlabel('거래 건수')
plt.ylabel('세그먼트')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

#### 5.2.3 세그먼트별 평균 구매금액

각 세그먼트의 1건당 평균 구매액

In [ ]:
# 세그먼트별 평균 구매금액
segment_avg = df.groupby('Segment')['Purchase'].mean().sort_values(ascending=False)

print("\n세그먼트별 평균 구매금액 Top 10:")
for i, (seg, avg) in enumerate(segment_avg.head(10).items(), 1):
    print(f"{i}. {seg}: ₹{avg:,.0f}")

# 시각화
plt.figure(figsize=(12, 6))
segment_avg.head(10).plot(kind='barh', color='#1abc9c')
plt.title('세그먼트별 평균 구매금액 Top 10', fontsize=14, fontweight='bold')
plt.xlabel('평균 구매금액 (₹)')
plt.ylabel('세그먼트')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

#### 5.2.4 매출 기여도 Top 5 세그먼트

총 매출액 기준 상위 5개 세그먼트 도출

In [ ]:
# Top 5 세그먼트 종합 분석
top5_segments = segment_sales.head(5).index

print("\n매출 기여도 Top 5 세그먼트 종합:")
print("=" * 80)
for i, seg in enumerate(top5_segments, 1):
    print(f"\n{i}. {seg}")
    print(f"   총 매출: ₹{segment_sales[seg]:,.0f} ({segment_sales_pct[seg]:.2f}%)")
    print(f"   거래 건수: {segment_count[seg]:,}건 ({segment_count_pct[seg]:.2f}%)")
    print(f"   평균 구매금액: ₹{segment_avg[seg]:,.0f}")
print("=" * 80)

### 5.3 캐시카우 세그먼트 식별

#### 5.3.1 최대 매출 기여 세그먼트 도출

1위 세그먼트의 성별, 연령대 확인

In [ ]:
# 캐시카우 세그먼트 식별
cashcow_segment = segment_sales.idxmax()
cashcow_gender = cashcow_segment.split('_')[0]
cashcow_age = cashcow_segment.split('_')[1]

print("\n" + "=" * 80)
print("🏆 캐시카우(Cash Cow) 세그먼트 식별 결과")
print("=" * 80)
print(f"세그먼트: {cashcow_segment}")
print(f"성별: {'남성' if cashcow_gender == 'M' else '여성'} ({cashcow_gender})")
print(f"연령대: {cashcow_age}세")
print("=" * 80)

#### 5.3.2 캐시카우 세그먼트 심층 분석

- 거래 건수
- 총 매출액
- 평균 구매금액
- 전체 매출 대비 비중(%)

In [ ]:
# 캐시카우 세그먼트 상세 분석
cashcow_data = df[df['Segment'] == cashcow_segment]

cashcow_count = len(cashcow_data)
cashcow_sales = cashcow_data['Purchase'].sum()
cashcow_avg = cashcow_data['Purchase'].mean()
cashcow_sales_ratio = (cashcow_sales / df['Purchase'].sum()) * 100
cashcow_count_ratio = (cashcow_count / len(df)) * 100

print("\n캐시카우 세그먼트 심층 분석:")
print("=" * 80)
print(f"거래 건수: {cashcow_count:,}건 (전체의 {cashcow_count_ratio:.2f}%)")
print(f"총 매출액: ₹{cashcow_sales:,.0f} (전체의 {cashcow_sales_ratio:.2f}%)")
print(f"평균 구매금액: ₹{cashcow_avg:,.0f}")
print(f"중앙값: ₹{cashcow_data['Purchase'].median():,.0f}")
print(f"표준편차: ₹{cashcow_data['Purchase'].std():,.0f}")
print("=" * 80)

#### 5.3.3 캐시카우 특성 분석

도시 등급, 결혼 여부, 거주 기간 등 추가 특성

In [ ]:
# 캐시카우 세그먼트의 추가 특성 분석
print("\n캐시카우 세그먼트 추가 특성:")
print("=" * 80)

# 도시 등급
print("\n도시 등급 분포:")
city_dist = cashcow_data['City_Category'].value_counts()
for city, count in city_dist.items():
    print(f"  {city}등급: {count:,}건 ({count/len(cashcow_data)*100:.1f}%)")

# 결혼 여부
print("\n결혼 여부:")
marital_dist = cashcow_data['Marital_Status'].value_counts()
for status, count in marital_dist.items():
    status_label = '기혼' if status == 1 else '미혼'
    print(f"  {status_label}: {count:,}건 ({count/len(cashcow_data)*100:.1f}%)")

# 거주 기간
print("\n현재 도시 거주 기간:")
stay_dist = cashcow_data['Stay_In_Current_City_Years'].value_counts().sort_index()
for stay, count in stay_dist.items():
    print(f"  {stay}년: {count:,}건 ({count/len(cashcow_data)*100:.1f}%)")

print("=" * 80)

### 5.4 베스트셀러 상품 분석

#### 5.4.1 캐시카우 세그먼트의 구매 제품 Top 10

해당 세그먼트가 가장 많이 구매한 제품 순위

In [ ]:
# 캐시카우 세그먼트의 제품별 구매 분석
cashcow_products = cashcow_data.groupby('Product_ID').agg({
    'Purchase': ['count', 'sum', 'mean']
}).reset_index()
cashcow_products.columns = ['Product_ID', 'Purchase_Count', 'Total_Sales', 'Avg_Price']
cashcow_products = cashcow_products.sort_values('Purchase_Count', ascending=False)

print("\n캐시카우 세그먼트의 구매 제품 Top 10:")
print("=" * 80)
for i, row in cashcow_products.head(10).iterrows():
    print(f"{i+1}. 제품코드: {row['Product_ID']}")
    print(f"   구매 건수: {int(row['Purchase_Count'])}건")
    print(f"   총 매출: ₹{row['Total_Sales']:,.0f}")
    print(f"   평균 가격: ₹{row['Avg_Price']:,.0f}")
    print("-" * 80)

#### 5.4.2 베스트셀러 상품 식별

1위 제품의 제품 코드, 구매 건수, 총 매출

In [ ]:
# 베스트셀러 상품 식별
best_product = cashcow_products.iloc[0]
best_product_id = best_product['Product_ID']
best_product_count = int(best_product['Purchase_Count'])
best_product_sales = best_product['Total_Sales']
best_product_avg = best_product['Avg_Price']

print("\n" + "=" * 80)
print("💎 베스트셀러 상품(Best Seller) 식별 결과")
print("=" * 80)
print(f"제품 코드: {best_product_id}")
print(f"구매 건수: {best_product_count:,}건")
print(f"총 매출: ₹{best_product_sales:,.0f}")
print(f"평균 가격: ₹{best_product_avg:,.0f}")
print("=" * 80)

#### 5.4.3 베스트셀러 상품 특성 분석

- 제품 카테고리
- 평균 가격
- 구매 빈도
- 총 매출 기여도

#### 5.4.4 효자 상품의 전체 고객 분포

다른 세그먼트에서의 구매 비중 확인

---

## 6. 핵심 인사이트

### 인사이트 1

[데이터 분석 결과] 성별 [X]세 고객이 전체 매출의 [Y]%를 차지하며 캐시카우로 식별되었다.

### 인사이트 2

캐시카우 세그먼트는 전체 거래의 [A]%를 차지하며, 평균 구매금액은 [B]원이다.

### 인사이트 3

캐시카우 세그먼트가 가장 많이 구매하는 효자 상품은 [제품코드]이며, 총 [C]건 구매되었다.

### 인사이트 4

효자 상품은 캐시카우 세그먼트 구매의 [D]%를 차지하며, 총 매출은 [E]원이다.

### 인사이트 5

캐시카우 세그먼트와 효자 상품을 결합한 타겟 마케팅 시 예상 매출 증대 효과는 약 [F]%이다.

---

## 7. 세그먼트별 비교 분석

### 7.1 상위 3개 세그먼트 비교

매출 Top 3 세그먼트의 특성 비교표

### 7.2 캐시카우의 차별화 요소

1위 세그먼트가 2위 대비 우위를 보이는 지표

### 7.3 효자 상품의 세그먼트별 선호도

효자 상품이 다른 세그먼트에서도 인기가 있는지 검증

---

## 8. 결론 및 제언

### 8.1 결론

데이터 분석 결과, 월마트에서 가장 매출을 많이 기여하는 **캐시카우(Cash Cow)는 [연령대] [성별]**이며, 이들이 가장 많이 구매하는 **효자 상품은 [제품코드]**이다.

이들을 공략하기 위해 **SNS 광고, 인플루언서 마케팅** 등을 통해 [효자 상품]을 집중 프로모션해야 하며, 예상 효과는 **매출 20% 증대**이다.

### 8.2 마케팅 전략 제언

#### 전략 1: 세그먼트 타겟 광고

[캐시카우 세그먼트] 대상 SNS 광고 집중 (Instagram, Facebook, YouTube)

#### 전략 2: 효자 상품 프로모션

[제품코드] 중심의 번들 상품, 할인 이벤트 기획

#### 전략 3: 인플루언서 마케팅

[캐시카우 성별/연령대] 영향력 있는 인플루언서와 [효자 상품] 협업

#### 전략 4: 크로스셀링 전략

효자 상품 구매자에게 연관 제품 추천 알고리즘 적용

#### 전략 5: 재고 및 진열 최적화

효자 상품의 재고 확대 및 매장 내 프라임 위치 배치

---

### 9.1 연구의 한계

- 시간 데이터 부재로 계절성 분석 불가
- 제품명 익명화로 실제 상품 특성 파악 제한
- 온라인/오프라인 구분 정보 부재

### 9.2 향후 연구 방향

- 시계열 데이터 확보 시 트렌드 분석
- 제품 카테고리별 심층 분석
- 고객 생애 가치(LTV) 분석

---

## 10. 부록

### 10.1 사용 라이브러리

pandas, numpy, matplotlib, seaborn

### 10.2 데이터 출처

Kaggle - Walmart Sales Dataset

GitHub Repository 링크 

---